### Import

In [1]:
import os
import sys
import pickle 
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt 
from sklearn.model_selection import train_test_split

### General parameters

In [2]:
path_timeseries = "Extraction/Data/Output/"
path_output = "./Data/EHR/"

### Reading Demographic Data

In [3]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'admission.csv')
    all_filenames.append(df_file)
    
df_admission = pd.concat([pd.read_csv(file, low_memory=False) for file in all_filenames if os.path.exists(file)])

In [4]:
df_admission.head(3)

### Reading Comorbidities Data

In [6]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'comorbidity.csv')
    all_filenames.append(df_file)
    
df_comorbidity = pd.concat([pd.read_csv(file, low_memory=False) for file in all_filenames if os.path.exists(file)])

In [7]:
df_comorbidity.head(3)

### Reading Missing Percentage

In [8]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'missing_percentage.csv')
    all_filenames.append(df_file)
    
df_missing = pd.concat([pd.read_csv(file, low_memory=False) for file in all_filenames if os.path.exists(file)])

In [9]:
df_missing.head(3)

### Reading Data

In [10]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in (all_stays):
    df_file = os.path.join(path_timeseries, str(stay_id), 'imputed_timeseries.csv')
    all_filenames.append(df_file)
    
df = pd.concat([pd.read_csv(file, nrows=72, low_memory=False) for file in all_filenames if os.path.exists(file)])
df = df[(df.Bins >= 0) & (df.Bins <= 23)]
df = df.reset_index(drop=True)

In [11]:
df.head(3)

### Reading Data based on Hours

In [5]:
def read_rows_startH_to_endH(path_timeseries, start=24, length=24, start_h=24, end_h=47):
    
    all_stays  = pd.Series(os.listdir(path_timeseries))
    all_filenames = []

    for stay_id in (all_stays):
        df_file = os.path.join(path_timeseries, str(stay_id), 'imputed_timeseries.csv')
        all_filenames.append(df_file)

    dfs = []
    for file in all_filenames:
        if os.path.exists(file):
            df = pd.read_csv(file, skiprows=range(1, start), nrows=length, low_memory=False)
            df = df[(df.Bins >= start_h) & (df.Bins <= end_h)]
            if not df.empty:
                dfs.append(df)

    final_df = pd.concat(dfs, ignore_index=True)
    return final_df

In [6]:
# second_day_df = read_rows_startH_to_endH(path_timeseries, start=24, length=48, start_h=24, end_h=47)
# second_day_df = second_day_df.reset_index(drop=True)

In [6]:
# third_day_df = read_rows_startH_to_endH(path_timeseries, start=48, length=48, start_h=48, end_h=71)
# third_day_df = third_day_df.reset_index(drop=True)

### Read Radiology Notes ID

In [9]:
note_columns = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'Bins', 
                'icu_expire_flag', 'hospital_expire_flag', 'radiology_note']

In [10]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'imputed_timeseries.csv')
    all_filenames.append(df_file)
    
df_note_ids = pd.concat([pd.read_csv(file, low_memory=False, usecols=note_columns) for file in all_filenames if os.path.exists(file)])

In [11]:
df_note_ids = df_note_ids[df_note_ids.radiology_note.notnull()]
df_note_ids = df_note_ids.drop_duplicates()
df_note_ids = df_note_ids.reset_index(drop=True)

df_note_ids = df_note_ids[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'Bins', 
                           'icu_expire_flag', 'hospital_expire_flag', 'radiology_note']]

In [12]:
df_note_ids.head(3)

### Read Radiology Images ID

In [13]:
cxr_columns = ['subject_id', 'hadm_id', 'stay_id', 'gender', 'age', 'race', 'Bins', 
               'icu_expire_flag', 'hospital_expire_flag', 'cxr_image']

In [14]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'imputed_timeseries.csv')
    all_filenames.append(df_file)
    
df_cxr_ids = pd.concat([pd.read_csv(file, low_memory=False, usecols=cxr_columns) for file in all_filenames if os.path.exists(file)])

In [15]:
df_cxr_ids = df_cxr_ids[df_cxr_ids.cxr_image.notnull()]
df_cxr_ids = df_cxr_ids.drop_duplicates()
df_cxr_ids = df_cxr_ids.reset_index(drop=True)

df_cxr_ids = df_cxr_ids[['subject_id', 'hadm_id', 'stay_id', 'gender', 'age', 'race', 'Bins', 
                         'icu_expire_flag', 'hospital_expire_flag', 'cxr_image']]

In [16]:
df_cxr_ids.head(3)

### Missing in 24H

In [7]:
missing_value_df = df.groupby('stay_id').apply(lambda x: x.isnull().all())
missing_value_df.drop(columns=['stay_id'], inplace=True)
missing_value_df = missing_value_df.reset_index()
missing_value_df = missing_value_df.replace(True, np.nan)
percent_missing = missing_value_df.isnull().sum() * 100 / len(missing_value_df)
missing_value_perc = pd.DataFrame({'column_name': missing_value_df.columns, 'percent_missing': percent_missing})
missing_value_perc.sort_values('percent_missing', inplace=True, ascending=False)
missing_value_perc.reset_index(inplace=True, drop=True)
missing_value_perc.head(3)

,column_name,percent_missing
0,Vancomycin (Peak),99.998600
1,CAM-ICU RASS LOC,99.713014
2,C-Reactive Protein (CRP),98.112891


### Save Data

In [12]:
# df_admission.to_csv(path_output + 'demographic_all.csv', index=False)
# df_comorbidity.to_csv(path_output + 'comorbidity_all.csv', index=False)
# df_missing.to_csv(path_output + 'missing_all.csv', index=False)
# df.to_csv(path_output + '0h_to_24h_data.csv', index=False)
# df_note_ids.to_csv(path_output + 'all_note_ids.csv', index=False)
# df_cxr_ids.to_csv(path_output + 'all_cxr_ids.csv', index=False)